## ***Initials***

In [26]:
import json
import pandas as pd
from bert_score import score
import pandas as pd

## ***Help Functions***

In [27]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

In [28]:
# extracts the municipal communities mentioned in the model outputs
def extract_mentioned_locations(text, valid_set):
    return [name for name in valid_set if name in text]

In [29]:
# return the match@k and precision@k for the model outputs
def evaluate_match_precision_recall(model_outputs, valid_locations, topk=3):
    match_at_k = []
    precision_at_k = []
    recall_at_k = []

    for entry in model_outputs:
        
        # Ground truth
        expected_output = entry["expected_completion"]

        # Extract the top-k true answers
        expected_names = extract_mentioned_locations(expected_output, valid_locations)
        expected_topk = set(expected_names[:topk])

        # Extract predicted communities from model output
        predicted = set(extract_mentioned_locations(entry["model_output"], valid_locations))

        # Match@k: at least one correct?
        match = int(len(predicted & expected_topk) > 0)

        # Precision@k: what % of predicted are correct
        precision = len(predicted & expected_topk) / len(predicted) if predicted else 0

        # Recall@k
        recall = len(predicted & expected_topk) / len(expected_topk) if expected_topk else 0

        match_at_k.append(match)
        precision_at_k.append(precision)
        recall_at_k.append(recall)

    return match_at_k, precision_at_k, recall_at_k

## ***Load the Data***

### ***Inference Outputs RAG***

In [30]:
inference_data = load_jsonl("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\8. LLM - RAG\\Extracted Files\\Inference Outputs\\inference_outputs_20.jsonl")

In [31]:
# Extract the model inference outputs
inference_outputs = [row["model_output"] for row in inference_data]

In [32]:
references = [row["expected_completion"] for row in inference_data]

### ***Neighborhoods***

In [33]:
# Load ELSTAT data
neighborhoods_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\3. Base Datasets\\3. Data -  Smaller Spatial Units\\1. Neighborhoods\\Extracted CSV Files\\neighborhoods_enriched.csv")

# Extract all unique neighborhoods
valid_communities = set(
    neighborhoods_df["Neighborhood"].dropna().unique()
)

print(f"Extracted {len(valid_communities)} valid neighborhoods.")

Extracted 48 valid neighborhoods.


## ***Compute: BERT-score***

In [34]:
P_l, R_l, F1_l = score(inference_outputs, references, lang="en", verbose=True)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/5 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 202.44 seconds, 0.79 sentences/sec


In [35]:
# === Collect Results ===
df = pd.DataFrame({
    "prompt": [row["prompt"] for row in inference_data],
    "LLM_output": inference_outputs,
    "reference": references,
    "LLM_score": F1_l.tolist(),
})

In [36]:
# === Summary ===
print("Average BERTScore Precision:")
print(f"{P_l.mean():.4f}")

print("Average BERTScore Recall:")
print(f"{R_l.mean():.4f}")

print("Average BERTScore F1:")
print(f"{F1_l.mean():.4f}")

Average BERTScore Precision:
0.8608
Average BERTScore Recall:
0.8290
Average BERTScore F1:
0.8446


## ***Compute: Match@k / Precision@k***

In [37]:
# Match@3, Precision@3, Recall@3
match, precision, recall = evaluate_match_precision_recall(inference_data, valid_communities)
print(f"LLM: Match@3 = {sum(match)/len(match):.2%}, Precision@3 = {sum(precision)/len(precision):.2%}, Recall@3 = {sum(recall)/len(recall):.2%}")

LLM: Match@3 = 71.07%, Precision@3 = 27.78%, Recall@3 = 28.93%


## ***Save the Metrics***

Here I will copy and paste the metrics produced for each LLM Fine-Tuning attempt:

***Attempt 1*** (3 epochs)

- Average BERTScore Precision: 0.8727
- Average BERTScore Recall: 0.8381
- Average BERTScore F1: 0.8550
- Match@3 = 83.02%
- Precision@3 = 36.69%
- Recall@3 = 36.69%

***Attempt 2*** (5 epochs)
- Average BERTScore Precision: 0.8648
- Average BERTScore Recall: 0.8314
- Average BERTScore F1: 0.8477
- Match@3 = 78.62%
- Precision@3 = 35.48%
- Recall@3 = 35.22%

***Attempt 3*** (8 epochs)

- Average BERTScore Precision: 0.8608
- Average BERTScore Recall: 0.8290
- Average BERTScore F1: 0.8446
- Match@3 = 71.07%
- Precision@3 = 27.78%
- Recall@3 = 28.93%